### Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

### Importing CSV file

In [ ]:
#importing the csv file
file_path = r'data_collection\data_collection\automobile.tn\cleaned_automobile_tn.csv'
full_df = pd.read_csv(file_path)
full_df.head()

## **Data Wrangling process**

### Checking null values & dropping unnecessary columns

In [ ]:
#dropping the first column
drop_column = full_df.columns.values.tolist()[0]
full_df = full_df.drop(columns=drop_column)

In [ ]:
null = full_df.isnull()
null.apply(pd.value_counts).fillna(0)

In [ ]:
#dropping interior, and links for being unrelevant
full_df = full_df.drop(columns=['interior','link'])

In [ ]:
#checking datatypes
df = full_df.drop(columns='description') #will use it later for feature engineering
df.info()

### Converting datatypes of columns

In [ ]:
#converting datatypes
df = df.astype('str')

#removing characters from numerical columns
def extract_number_regex(text):
    try:
        match = re.search(r'(\d+\.?\d*)', text.replace(',', '').replace(' ', ''))
        if match:
            return int(float(match.group(1)))
    except:
        pass
    return None

for col in ['price','mileage','engine-size','fiscal-power']:
    df[col] = df[col].map(extract_number_regex)

In [ ]:
#changing datetype
def extract_date_regex(text): #triple date entry
    try:
        match = re.search(r'^\d{2}\.\d{2}\.\d{4}$', text)
        if match:
            return datetime.strptime(text, '%d.%m.%Y').date()
    except:
        pass
    return None

df['publish-date'] = df['publish-date'].map(extract_date_regex)

def extract_date_month_regex(text): #double date entry
    try:
        match = re.search(r'^\d{1,2}\.\d{4}$', text)
        if match:
            return datetime.strptime(text, '%m.%Y').date()
    except:
        pass
    return None

df['circulation-date'] = df['circulation-date'].map(extract_date_month_regex)

In [ ]:
#change columns content to lower cases
string_columns = ['title','brand','model','fuel','gear','body-type','location']
for column in string_columns:
    df[column] = df[column].map(lambda x: x.lower())

### Replacing NaN values using basic imputation techniques and machine learning predictions

In [ ]:
#replacing NaN values
df['publish-date'] = pd.to_datetime(df['publish-date'])
avg_publish_date = df['publish-date'].mean().date()
df['publish-date'] = df['publish-date'].replace(np.NaN, avg_publish_date)
avg_publish_date

In [ ]:
#setting the car age in months
df['circulation-date'] = pd.to_datetime(df['circulation-date'])
df['publish-date'] = pd.to_datetime(df['publish-date'])

avg_publish_date = df['publish-date'].mean().date()
df['car-age'] = df['circulation-date'].apply(lambda x: (avg_publish_date - x.date()).days //30.42 if pd.notna(x) else np.nan)

In [ ]:
#replacing some missing values in engine-size
df['engine-size'] = df.groupby(['brand','model','fuel','gear'])['engine-size'].transform(lambda x: x.fillna(x.median()))

In [ ]:
temp = df.isnull()
temp.apply(pd.value_counts).fillna(0)

In [ ]:
df['location'].replace('nan', np.nan, inplace=True)

In [ ]:
#dealing with nan values on location, using decision tree classifier
from sklearn.tree import DecisionTreeClassifier

known = df[df['location'].notna()]
unkown = df[df['location'].isna()]

#training feature
X = known[['price','brand','model','body-type']]
y = known['location']

#features encoding

X = pd.get_dummies(X)
X_unkown = pd.get_dummies(unkown[['price','brand','location','body-type']])
X_unkown = X_unkown.reindex(columns=X.columns, fill_value=0)  #making sure we have the same columns order

clf = DecisionTreeClassifier()
clf.fit(X,y)
df.loc[df['location'].isna(), 'location'] = clf.predict(X_unkown)

In [ ]:
#replacing eletrical engine size with 0 instead of NaN
df.loc[df['fuel'] == 'electrique','engine-size'] = df.loc[df['fuel'] == 'electrique', 'engine-size'].fillna(0)

In [ ]:
#building a regression model to predict car engine size

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,    
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

known = df[df['engine-size'].notna()]
unkown = df[df['engine-size'].isna()]

X_categorical = pd.get_dummies(known[['brand', 'model', 'fuel','gear','body-type']])
X_numerical = known[['price','fiscal-power']]

X = pd.concat([X_categorical, X_numerical], axis=1)
y = known['engine-size']

X_categorical_unkown = pd.get_dummies(unkown[['brand', 'model', 'fuel','gear','body-type']])
X_numerical_unkown = unkown[['price','fiscal-power']]

X_unkown = pd.concat([X_categorical_unkown, X_numerical_unkown], axis=1)
X_unkown = X_unkown.reindex(columns=X.columns, fill_value=0)

model.fit(X,y)
df.loc[df['engine-size'].isna(), 'engine-size'] = model.predict(X_unkown)

In [ ]:
df[['engine-size']] = df[['engine-size']].round()

In [ ]:
#some dates are not reliable, pandas allows .median() to work on datetime columns
df['circulation-date'] = df.groupby('model')['circulation-date'].transform(lambda x: x.fillna(x.median()))

In [ ]:
#some dates are not reliable
df = df.dropna(subset=['circulation-date'])

### Adding a new car age feature for better interpretation of circulation dates

In [ ]:
#rounding up the engine sizes to the nearest 100 standardize values
df['engine-size'] = df['engine-size'].round(-2)

### Data Wrangling Evaluation

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
for col in ['fuel','brand','location','gear','body-type']:
    print(col, df[col].unique())

In [ ]:
df.hist(figsize=(12, 10))

In [ ]:
df.corr(numeric_only=True)

In [ ]:
#checking random samples
df.sample(5).T

In [ ]:
#checking for outliers per each column
import seaborn as sns
sns.boxplot(x=df['car-age'])

### dropping rows with non logical mileage

In [ ]:
df = df.drop(index=[1951,723])

### **SAVING THE DATAFRAME TO A CSV FILE**

In [ ]:
df.to_csv('automobiletn_set.csv', index=False)